# 01 – Exploração da ABT

Análise exploratória da população de estudo, a partir da cópia congelada da ABT (`data/processed/abt_tuberculose.csv`, gerada por `src/features/construir_abt.py`). O notebook não precisa do banco de dados.

Execute a partir da pasta `ia/notebooks`. **Antes de fazer commit, limpe as saídas** (*Clear All Outputs*): o notebook só mostra resultados agregados, mas é melhor que o Git não guarde nada derivado dos microdados.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

RAIZ = Path.cwd().parent          # o notebook fica em ia/notebooks
ALVO = "alvo_preditivo"           # 1 = perda de seguimento | 0 = cura

abt = pd.read_csv(RAIZ / "data" / "processed" / "abt_tuberculose.csv", sep=";")
categoricas = ["faixa_etaria", "sexo", "teste_hiv", "alcool", "tabagismo", "drogas",
               "escolaridade", "raca", "faixa_populacao", "regiao"]
numericas = ["idhm", "indice_gini", "cobertura_aps_2021"]
print(f"{len(abt):,} notificações | {abt.shape[1]} colunas".replace(",", "."))

## 1. Desfecho

In [ ]:
resumo = abt[ALVO].map({0: "Cura", 1: "Perda de seguimento"}).value_counts().to_frame("n")
resumo["%"] = (resumo["n"] / resumo["n"].sum() * 100).round(1)
resumo

## 2. Perda de seguimento por ano de notificação

O teste é o último ano (2022). A proporção de perda maior nesse ano é uma das razões para a divisão temporal.

In [ ]:
por_ano = abt.groupby("ano_notificacao")[ALVO].agg(casos="size", perda="mean")
por_ano["perda"] = (por_ano["perda"] * 100).round(1)
display(por_ano)

ax = por_ano["perda"].plot.bar(color="#2a78d6", figsize=(6, 3), rot=0)
ax.set_ylabel("Perda de seguimento (%)")
ax.set_xlabel("Ano de notificação")
plt.show()

## 3. Proporção de "Ignorado" por variável

In [ ]:
ignorado = (abt[categoricas] == "Ignorado").mean().mul(100).round(1).sort_values(ascending=False)
ignorado.to_frame("% Ignorado")

## 4. Taxa de perda de seguimento por categoria

In [ ]:
def taxa_por_categoria(var):
    t = abt.groupby(var)[ALVO].agg(n="size", perda="mean")
    t["perda (%)"] = (t.pop("perda") * 100).round(1)
    return t

for var in categoricas:
    print(f"\n=== {var} ===")
    display(taxa_por_categoria(var))

## 5. Indicadores municipais por desfecho

In [ ]:
abt.groupby(ALVO)[numericas].describe().T.round(3)

In [ ]:
fig, eixos = plt.subplots(1, len(numericas), figsize=(12, 3))
for ax, var in zip(eixos, numericas):
    for valor, rotulo, cor in [(0, "Cura", "#2a78d6"), (1, "Perda de seguimento", "#eb6834")]:
        abt.loc[abt[ALVO] == valor, var].plot.hist(ax=ax, bins=30, density=True, alpha=0.5,
                                                   color=cor, label=rotulo)
    ax.set_title(var)
eixos[0].legend()
plt.tight_layout()
plt.show()

## 6. Valores faltantes nos indicadores municipais

Os faltantes são imputados pela mediana dentro do *pipeline* de cada modelo, nunca na ABT.

In [ ]:
abt[numericas].isna().sum().to_frame("faltantes")